<a href="https://colab.research.google.com/github/vikivinieratou-ship-it/Housing-Price-Prediction-Linear-Regression---R/blob/main/Price_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
!pip install kaggle
import os
print(os.listdir())



['.config', 'data.csv', 'sample_data']


In [16]:
import os
print(os.listdir())

df = pd.read_csv("data.csv")

['.config', 'data.csv', 'sample_data']


In [21]:
import pandas as pd
import numpy as np

# 1. Load data
df = pd.read_csv("data.csv")

# 2. Clean column names
df.columns = df.columns.str.lower()

# 3. Handle missing values
df = df.dropna()

# 4. Use smaller sample for speed
df = df.sample(10000, random_state=42)

# 5. Split features / target
X = df.drop(["price", "transactionid", "customerid", "productid"], axis=1)
y = df["price"]

# 6. Feature engineering
X["value_per_item"] = df["price"] / (df["quantity"] + 1)

X["loyalty_income"] = df["customerloyaltyscore"] * (
    df["customerincomegroup"] == "High"
).astype(int)

X["age_group"] = pd.cut(
    df["customerage"],
    bins=[0, 25, 50, 100],
    labels=[0, 1, 2]
)

# 7. Encoding (safe + efficient)
X = pd.get_dummies(X, drop_first=True)

# 8. Train/test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 9. Fast + good model
from sklearn.ensemble import GradientBoostingRegressor

model = GradientBoostingRegressor(
    n_estimators=50,
    max_depth=3
)

model.fit(X_train, y_train)

# 10. Evaluation
from sklearn.metrics import mean_absolute_error

preds = model.predict(X_test)

mae = mean_absolute_error(y_test, preds)

print("MAE:", mae)

MAE: 12.465661389873949


In [23]:
import pickle

with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

print("Model saved!")

Model saved!


In [24]:
import os
print(os.listdir())

['.config', 'data.csv', 'model.pkl', 'sample_data']


In [25]:
from fastapi import FastAPI
import pickle
import numpy as np

app = FastAPI()

# load model
with open("model.pkl", "rb") as f:
    model = pickle.load(f)

@app.get("/")
def home():
    return {"message": "ML API running"}

@app.post("/predict")
def predict(data: dict):
    features = np.array(list(data.values())).reshape(1, -1)
    prediction = model.predict(features)
    return {"prediction": prediction.tolist()}

In [26]:
feature_names = X.columns

In [27]:
import pickle

pickle.dump(feature_names, open("features.pkl", "wb"))

In [28]:
from fastapi import FastAPI
import pickle
import pandas as pd

app = FastAPI()

model = pickle.load(open("model.pkl", "rb"))
features = pickle.load(open("features.pkl", "rb"))

@app.post("/predict")
def predict(data: dict):
    df = pd.DataFrame([data])
    df = pd.get_dummies(df)

    # align columns
    df = df.reindex(columns=features, fill_value=0)

    prediction = model.predict(df)
    return {"prediction": prediction.tolist()}

In [29]:
test_data = {
    "customerage": 30,
    "customergender": "Female",
    "customerincomegroup": "High",
    "customerloyaltyscore": 50,
    "quantity": 2
}

print(predict(test_data))

{'prediction': [19.660046630194934]}


In [30]:
from google.colab import files

files.download("model.pkl")
files.download("features.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>